# Download EUMETSAT data

In [7]:
import os
os.environ["EUMETSAT_CONSUMER_KEY"] ="rN0eRwPs9ibZvRTPCFk2ICqYXvoa"
os.environ["EUMETSAT_CONSUMER_SECRET"] ="nlylzM25OzvOHGWf_M3Ka5Q_u4Qa"

In [12]:
import os
import shutil
import re
import time
import datetime as dt
import subprocess
from pathlib import Path

import eumdac


In [ ]:


COLLECTION_ID = "EO:EUM:DAT:MSG:HRSEVIRI"

# La tua ROI
N, S, W, E = 48, 30, -7, 46

def run(cmd):
    """Esegue un comando e ritorna stdout (raise se errore)."""
    p = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return p.stdout.strip()

def extract_job_id(output: str) -> str:
    """
    Estrae un JobID dall'output di 'eumdac tailor post'.
    Formati possibili cambiano: quindi faccio match robusto su token alfanumerico-lungo.
    Se non lo trova: stampa output e alza errore.
    """
    # Prova pattern tipo UUID
    m = re.search(r"\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b", output)
    if m:
        return m.group(0)
    # Fallback: prima “word” lunga (alcuni job id non sono uuid)
    m = re.search(r"\b[a-zA-Z0-9_-]{12,}\b", output)
    if m:
        return m.group(0)
    raise RuntimeError(f"Non riesco a riconoscere il JobID dall'output:\n{output}")

def wait_job(job_id: str, poll_s: int = 10, timeout_s: int = 1800):
    """
    Attende che il job finisca.
    Uso 'eumdac tailor status <jobid>' se disponibile.
    Se il tuo eumdac usa un comando diverso, fai: 'eumdac tailor --help'.
    """
    t0 = time.time()
    while True:
        if time.time() - t0 > timeout_s:
            raise TimeoutError(f"Timeout: job {job_id} non finito entro {timeout_s}s")

        try:
            status_out = run(["eumdac", "tailor", "status", job_id])
        except subprocess.CalledProcessError as e:
            # Alcune versioni potrebbero usare 'describe' o 'show'
            raise RuntimeError(
                "Il comando 'eumdac tailor status' non ha funzionato nella tua versione.\n"
                "Esegui nel terminale:  eumdac tailor --help\n"
                "e adatta il comando di polling.\n\n"
                f"stderr:\n{e.stderr}"
            )

        # Heuristica: cerca parole tipiche
        low = status_out.lower()
        if "finished" in low or "completed" in low or "done" in low or "success" in low:
            return status_out
        if "failed" in low or "error" in low:
            raise RuntimeError(f"Job {job_id} fallito:\n{status_out}")

        print(f"[{job_id}] in corso... ({poll_s}s)")
        time.sleep(poll_s)

def main():
    consumer_key = os.getenv("EUMETSAT_CONSUMER_KEY")
    consumer_secret = os.getenv("EUMETSAT_CONSUMER_SECRET")
    if not consumer_key or not consumer_secret:
        raise SystemExit("Imposta EUMETSAT_CONSUMER_KEY e EUMETSAT_CONSUMER_SECRET.")

    # 1) Cerca prodotti (qui esempio: 1 ora)
    start = dt.datetime(2026, 1, 16, 0, 0)
    end   = dt.datetime(2026, 1, 23, 0, 0)

    token = eumdac.AccessToken((consumer_key, consumer_secret))
    datastore = eumdac.DataStore(token)
    collection = datastore.get_collection(COLLECTION_ID)

    products = collection.search(dtstart=start, dtend=end)
    product_ids = [str(p) for p in products]

    print(f"Trovati {len(product_ids)} prodotti in {start} → {end}")
    if not product_ids:
        return

    out_dir = Path("./hrseviri_roi")
    out_dir.mkdir(parents=True, exist_ok=True)

    # 2) Per ogni prodotto: Data Tailor ROI extraction + download
    for pid in product_ids:
        # Chain Data Tailor:
        # - product: HRSEVIRI
        # - format: (esempio) geotiff
        # - roi: NSWE (North, South, West, East)
        #
        # L'esempio base "product+format" è documentato; ROI è una funzione tipica del Data Tailor. :contentReference[oaicite:3]{index=3}
        chain = f"product: HRSEVIRI, format: geotiff, roi: NSWE[{N},{S},{W},{E}]"

        print(f"\nSubmitting tailor job for product: {pid}")
        post_out = run([
            "eumdac", "tailor", "post",
            "-c", COLLECTION_ID,
            "-p", pid,
            "--chain", chain
        ])
        print(post_out)

        job_id = extract_job_id(post_out)
        print("JobID:", job_id)

        # 3) Attendi fine job e scarica risultato
        wait_job(job_id, poll_s=10, timeout_s=3600)

        # download tailored product
        # (La CLI supporta 'tailor download <JobID> -o <path>' in esempi ufficiali.) :contentReference[oaicite:4]{index=4}
        print(f"Downloading job {job_id} → {out_dir}")
        run(["eumdac", "tailor", "download", job_id, "-o", str(out_dir)])

    print("\nDone.")


In [16]:
# -------------------------
# PARAMETRI
# -------------------------

#COLLECTION_ID = "EO:EUM:DAT:MSG:HRSEVIRI"  # ogni 15 min, full disk (non rapid scan)
COLLECTION_ID = "EO:EUM:DAT:MSG:MSG15-RSS"  # ogni 5 min, ma solo su una porzione di disco (non tutto)

start = dt.datetime(2026, 1, 16, 0, 0)
end   = dt.datetime(2026, 1, 23, 0, 0)

LIMIT = None          # numero massimo prodotti da scaricare (metti None per tutti)
OUTDIR = Path("/media/isacDisk2/from_eumetsat/seviri_rapidscan")
OUTDIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# CREDENZIALI
# -------------------------

consumer_key = os.getenv("EUMETSAT_CONSUMER_KEY")
consumer_secret = os.getenv("EUMETSAT_CONSUMER_SECRET")

if not consumer_key or not consumer_secret:
    raise RuntimeError("Imposta EUMETSAT_CONSUMER_KEY e EUMETSAT_CONSUMER_SECRET")

# -------------------------
# CONNESSIONE
# -------------------------

token = eumdac.AccessToken((consumer_key, consumer_secret))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection(COLLECTION_ID)

# -------------------------
# SEARCH
# -------------------------

products = collection.search(dtstart=start, dtend=end)

selected_ids = []
for p in products:
    selected_ids.append(str(p))     # product_id
    if LIMIT and len(selected_ids) >= LIMIT:
        break

print(f"Trovati {len(selected_ids)} prodotti da scaricare")

# Download vero: get_product(...) + open() + copy
for i, product_id in enumerate(selected_ids, 1):
    print(f"[{i}/{len(selected_ids)}] Retrieving {product_id}")

    product = datastore.get_product(product_id=product_id, collection_id=COLLECTION_ID)    

    with product.open() as fsrc:
        out_path = OUTDIR / fsrc.name

        # evita riscarico se già esiste
        if out_path.exists() and out_path.stat().st_size > 0:
            print(f"  -> già presente: {out_path.name}, skip")
            continue

        print(f"  -> downloading {fsrc.name}")
        with open(out_path, "wb") as fdst:
            shutil.copyfileobj(fsrc, fdst)

        print(f"  -> done: {out_path.name} ({out_path.stat().st_size/1e6:.1f} MB)")

print("Download completato.")

Trovati 1988 prodotti da scaricare
[1/1988] Retrieving MSG4-SEVI-MSG15-0100-NA-20260122235917.968000000Z-NA
  -> downloading MSG4-SEVI-MSG15-0100-NA-20260122235917.968000000Z-NA.zip
  -> done: MSG4-SEVI-MSG15-0100-NA-20260122235917.968000000Z-NA.zip (36.4 MB)
[2/1988] Retrieving MSG4-SEVI-MSG15-0100-NA-20260122235418.231000000Z-NA
  -> downloading MSG4-SEVI-MSG15-0100-NA-20260122235418.231000000Z-NA.zip
  -> done: MSG4-SEVI-MSG15-0100-NA-20260122235418.231000000Z-NA.zip (36.5 MB)
[3/1988] Retrieving MSG4-SEVI-MSG15-0100-NA-20260122234919.700000000Z-NA
  -> downloading MSG4-SEVI-MSG15-0100-NA-20260122234919.700000000Z-NA.zip
  -> done: MSG4-SEVI-MSG15-0100-NA-20260122234919.700000000Z-NA.zip (36.5 MB)
[4/1988] Retrieving MSG4-SEVI-MSG15-0100-NA-20260122234419.963000000Z-NA
  -> downloading MSG4-SEVI-MSG15-0100-NA-20260122234419.963000000Z-NA.zip
  -> done: MSG4-SEVI-MSG15-0100-NA-20260122234419.963000000Z-NA.zip (36.6 MB)
[5/1988] Retrieving MSG4-SEVI-MSG15-0100-NA-20260122233920.225000